In [ ]:
#Installation + Restart automatique pour que Python détecte le package installé

import os, sys

try:
    import dllm
    print("dllm déjà installé.")

except ImportError:
    print("Installation de dllm depuis GitHub.")
    os.system("git clone https://github.com/ZHZisZZ/dllm.git 2>/dev/null || true") #2>/dev/null || true permet d'ignorer l'erreur si le git clone existe déjà
    os.system("pip install -e dllm/ -q")
    os.system("pip install torch transformers accelerate datasets -q")
    os.system("pip install evaluate rouge_score bert_score sacrebleu -q")

    print("Installation terminée !")
    print("Restart automatique du kernel.") #Important pour que Python prenne en compte les installations

    import time
    time.sleep(3)

    # Relance de python
    os.kill(os.getpid(), 9)

dllm déjà installé.


In [ ]:
# Imports & chargement MDLM
import torch
import numpy as np
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForMaskedLM

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")

MODEL_NAME = "dllm-hub/Qwen3-0.6B-diffusion-mdlm-v0.1"

#Tokenizer : on passe de phrase à des tokens
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

#Chargement du modèle
model = AutoModelForMaskedLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
).to(device).eval()

# Pad pour égaliser la longueur des phrases avec [PAD] ou le token de fin de phrase eos_token
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
pad_id = tokenizer.pad_token_id
mask_id = tokenizer.mask_token_id #Pour masquer des mots que l'IA doit deviner

print(f"Modèle chargé | mask_token_id={mask_id} | pad_token_id={pad_id}")

Device : cpu


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

The repository dllm-hub/Qwen3-0.6B-diffusion-mdlm-v0.1 contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/dllm-hub/Qwen3-0.6B-diffusion-mdlm-v0.1 .
 You can inspect the repository content at https://hf.co/dllm-hub/Qwen3-0.6B-diffusion-mdlm-v0.1.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


modeling_qwen3.py: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/157 [00:00<?, ?B/s]

Modèle chargé | mask_token_id=151669 | pad_token_id=151643


In [ ]:
# Fonctions MDLM (à ne PAS modifier) -> pris de Hugging Face

def add_gumbel_noise(logits, temperature):
    """
    Ajoute du bruit mathématique (Gumbel) aux prédictions du modèle.
    But : Éviter que le modèle génère des phrases trop plates ou répétitives.
    Plus la 'temperature' est haute, plus on force le modèle à être créatif
    (en choisissant parfois des mots moins probables).
    """
    if temperature == 0:
        return logits
    logits = logits.to(torch.float64) # Haute précision pour les calculs de probabilités
    noise = torch.rand_like(logits, dtype=torch.float64)
    gumbel_noise = (-torch.log(noise)) ** temperature
    return logits.exp() / gumbel_noise


def get_num_transfer_tokens(mask_index, steps):
    """
    Agit comme un "planificateur".
    But : Calculer combien de jetons [MASK] on a le droit de transformer en vrais mots à chaque étape.
    Exemple : S'il y a 64 masques et 8 étapes, la fonction dira au modèle
    de figer exactement 8 mots par étape (le 'base') pour avancer progressivement.
    """
    mask_num = mask_index.sum(dim=1, keepdim=True)
    base = mask_num // steps
    remainder = mask_num % steps
    num_transfer_tokens = (
        torch.zeros(mask_num.size(0), steps, device=mask_index.device, dtype=torch.int64) + base
    )
    for i in range(mask_num.size(0)):
        num_transfer_tokens[i, :remainder[i]] += 1
    return num_transfer_tokens


@torch.no_grad() # Désactive l'apprentissage (rétropropagation) pour économiser la mémoire RAM
def generate_mdlm(model, prompt, prompt_lens, pad_id,
                  steps=64, max_new_tokens=128, block_size=64,
                  temperature=0.0, cfg_scale=0.0, remasking="low_confidence"):
    """
    Le moteur principal de génération MDLM.
    Il part d'un texte rempli de masques et les révèle petit à petit.
    """
    batch_size = prompt.size(0)
    total_length = int(prompt_lens.max().item()) + max_new_tokens

    # 1. INITIALISATION DU TENSEUR GÉANT
    # On crée une grille vide remplie de pad_id
    x = torch.full((batch_size, total_length), pad_id, dtype=torch.long, device=device)

    # On y injecte la question (prompt) et on remplit la zone de réponse avec des [MASK]
    for i, length in enumerate(prompt_lens.tolist()):
        x[i, :length] = prompt[i, :length]              # Zone fixe (La question)
        x[i, length:length + max_new_tokens] = mask_id  # Zone à générer (Que des masques !)

    # 2. VERROUILLAGE DE LA QUESTION
    # On crée un masque booléen (True/False) pour ne jamais modifier les mots de la question
    prompt_index = (
        torch.arange(total_length, device=device).unsqueeze(0)
        < prompt_lens.unsqueeze(1)
    )
    positions = torch.arange(total_length, device=device)

    # 3. DÉBRUITAGE PAR BLOCS (Le cœur de la diffusion)
    # Pour ne pas saturer la mémoire, on démasque par blocs (ex: 64 tokens par 64 tokens)
    assert max_new_tokens % block_size == 0, \
        f"max_new_tokens ({max_new_tokens}) doit être divisible par block_size ({block_size})"
    num_blocks = max_new_tokens // block_size
    assert steps % num_blocks == 0
    steps_per_block = steps // num_blocks

    for num_block in range(num_blocks):
        block_start = prompt_lens + num_block * block_size
        block_end = block_start + block_size

        init_block_mask = (
            (positions.unsqueeze(0) >= block_start.unsqueeze(1))
            & (positions.unsqueeze(0) < block_end.unsqueeze(1))
            & (x == mask_id)
        )

        # On demande au planificateur combien de mots révéler à ce tour-ci
        num_transfer_tokens = get_num_transfer_tokens(init_block_mask, steps_per_block)

        # LA BOUCLE DE DIFFUSION (Génère -> Évalue la confiance -> Remasque)
        for i in range(steps_per_block):
            block_mask = (
                (positions.unsqueeze(0) >= block_start.unsqueeze(1))
                & (positions.unsqueeze(0) < block_end.unsqueeze(1))
                & (x == mask_id)
            )

            # A. Prédiction
            if cfg_scale > 0.0:
                un_x = x.clone()
                un_x[prompt_index] = mask_id
                x_ = torch.cat([x, un_x], dim=0)
                logits = model(x_).logits
                logits, un_logits = torch.chunk(logits, 2, dim=0)
                logits = un_logits + (cfg_scale + 1.0) * (logits - un_logits)
            else:
                logits = model(x).logits # Le modèle devine ce qui se cache sous les masques

            # B. Ajout de créativité et choix du mot
            logits_with_noise = add_gumbel_noise(logits, temperature=temperature)
            x0 = torch.argmax(logits_with_noise, dim=-1) # x0 contient les mots choisis

            # C. Calcul de la confiance (Le "remasking")
            if remasking == "low_confidence":
                # Le modèle calcule son pourcentage de certitude (ex: 95% sûr que c'est ce mot)
                p = F.softmax(logits, dim=-1)
                x0_p = torch.gather(p, dim=-1, index=x0.unsqueeze(-1)).squeeze(-1)
            elif remasking == "random":
                x0_p = torch.rand_like(x0, dtype=torch.float)
            else:
                raise NotImplementedError(remasking)

            # D. Validation et Remasquage
            confidence = torch.full_like(x0_p, -np.inf)
            confidence = torch.where(block_mask, x0_p, confidence)
            x0 = torch.where(block_mask, x0, x)

            # On cherche les "k" mots avec la plus haute confiance pour ce tour
            transfer_index = torch.zeros_like(x0, dtype=torch.bool)
            for j in range(confidence.shape[0]):
                k = int(num_transfer_tokens[j, i].item())
                if k == 0:
                    continue
                # On valide définitivement les k meilleurs mots
                _, select_index = torch.topk(confidence[j], k=k)
                transfer_index[j, select_index] = True

            # Les mots validés remplacent les [MASK] dans notre grille principale (x)
            # Les autres mots redeviennent des [MASK] pour la boucle suivante
            x[transfer_index] = x0[transfer_index]

    return x

In [ ]:
def generate_descriptions_mdlm(examples, n_samples=3, max_new_tokens=128,
                                steps=64, block_size=64, temperature=0.0):
    """
    Génère n_samples descriptions pour chaque exemple ToTTo avec MDLM.
    """
    all_predictions = []

    for idx, example in enumerate(examples):

        # 1. RÉCUPÉRATION DES DONNÉES
        table_text = example["serialized_table"]

        # 2. CONSTRUCTION DU PROMPT (Format Chat)
        conversation = [
            {"role": "system", "content": "You are a data-to-text assistant. Given a table, write a concise factual description."},
            {"role": "user",   "content": f"Table:\n{table_text}\n\nDescription:"}
        ]

        # 3. TOKENISATION BLINDÉE (Garantie 1D)
        # Étape A : On génère le texte brut avec les balises
        prompt_str = tokenizer.apply_chat_template(
            conversation,
            add_generation_prompt=True,
            tokenize=False
        )

        # Étape B : On demande explicitement un tenseur, et on le force en liste plate (1D)
        inputs = tokenizer(prompt_str, add_special_tokens=False, return_tensors="pt")
        prompt_ids = inputs["input_ids"].view(-1).tolist() # Force l'aplatissement (flatten)

        encoded = [prompt_ids]

        # Étape C : Création du tenseur (les dimensions seront parfaites)
        prompt_lens = torch.tensor([len(e) for e in encoded], dtype=torch.long, device=device)
        max_prompt_len = max(prompt_lens).item()
        prompt_tensor = torch.full((len(encoded), max_prompt_len), pad_id, dtype=torch.long, device=device)

        for i, ids in enumerate(encoded):
            prompt_tensor[i, :len(ids)] = torch.tensor(ids, dtype=torch.long, device=device)

        # 4. BOUCLE DE GÉNÉRATION
        sample_descriptions = []
        for sample_idx in range(n_samples):
            current_temp = temperature if sample_idx > 0 else 0.0

            out = generate_mdlm(
                model, prompt_tensor, prompt_lens, pad_id,
                steps=steps,
                max_new_tokens=max_new_tokens,
                block_size=block_size,
                temperature=current_temp,
                remasking="low_confidence"
            )

            # 5. DÉCODAGE
            new_tokens = out[0, prompt_lens[0]:prompt_lens[0] + max_new_tokens].tolist()
            text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
            sample_descriptions.append(text)

        all_predictions.append(sample_descriptions)

        if (idx + 1) % 10 == 0:
            print(f"  [{idx+1}/{len(examples)}] ✅ {sample_descriptions[0][:80]}...")

    return all_predictions

In [ ]:
# CELLULE 4 : Génération conditionnelle depuis les tableaux ToTTo

"""def generate_descriptions_mdlm(examples, n_samples=3, max_new_tokens=128,
                                steps=64, block_size=64, temperature=0.0):
  """
  Orchestre la génération de plusieurs descriptions pour un lot d'exemples ToTTo.
  Utilise le modèle de diffusion MDLM précédemment chargé.

  Args:
    examples : Liste de dictionnaires contenant les données ToTTo.
    n_samples : Nombre de descriptions différentes à générer par tableau.
    temperature : Degré de créativité pour les samples 2 et 3.

  Returns:
    all_predictions : Liste de listes de chaînes de caractères (ex: [[desc1, desc2, desc3], ...])
  """
  all_predictions = []

  for idx, example in enumerate(examples):
      # Récupération des données préparées à l'étape 1
      table_text = example["serialized_table"]

      # 2. Construction du prompt au format chat
      messages = [[
          {"role": "system", "content": "You are a data-to-text assistant. Given a table, write a concise factual description."},
           {"role": "user",   "content": f"Table:\n{table_text}\n\nDescription:"},
      ]]

      # 3. Tokenisation et Padding
      encoded = [
          tokenizer.apply_chat_template(
              m, add_generation_prompt=True,
              tokenize=True, enable_thinking=False
          )
          for m in messages
      ]
      prompt_lens = torch.tensor([len(e) for e in encoded], dtype=torch.long, device=device)
      max_prompt_len = max(prompt_lens).item()
      prompt_tensor = torch.full((len(encoded), max_prompt_len), pad_id, dtype=torch.long)

      for i, ids in enumerate(encoded):
          prompt_tensor[i, :len(ids)] = torch.tensor(ids, dtype=torch.long)
      prompt_tensor = prompt_tensor.to(device)

      # 4. Génération de n_samples descriptions
      sample_descriptions = []
      for sample_idx in range(n_samples):
        out = generate_mdlm(
            model, prompt_tensor, prompt_lens, pad_id,
            steps=steps,
            max_new_tokens=max_new_tokens,
            block_size=block_size,
            temperature=temperature if sample_idx > 0 else 0.0, # temperature=0 : déterministe pour le 1er, un peu de bruit ensuite
            remasking="low_confidence"
        )
        # Extraction uniquement des tokens générés (après le préfixe)
        new_tokens = out[0, prompt_lens[0]:prompt_lens[0] + max_new_tokens].tolist()
        text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        sample_descriptions.append(text)

        all_predictions.append(sample_descriptions)

        if (idx + 1) % 10 == 0:
            print(f"  [{idx+1}/{len(examples)}] ✅ {sample_descriptions[0][:80]}...")

  return all_predictions"""

In [ ]:
!pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=96f8d79feeebb6199da82001b2d3890a86df690d8eb233a136d4bfc5c722309c
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [ ]:
# CELLULE 5 : Calcul des métriques (Aligné avec l'équipe GPT-2)

import torch
import torch.nn.functional as F
from nltk.tokenize import word_tokenize
from nltk.translate.bleu_score import sentence_bleu
from rouge_score import rouge_scorer
import numpy as np
from transformers import AutoTokenizer, AutoModel

# 1. Configuration pour BERTScore (Identique à l'équipe GPT-2)
print("Chargement des modèles pour BERTScore...")
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
bert_model = AutoModel.from_pretrained("bert-base-uncased").to(device).eval()

def get_embeddings_batch(texts):
    inputs = bert_tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = bert_model(**inputs)

    return outputs.last_hidden_state  # (B, seq_len, hidden)

def bertscore_batch(preds, refs):
    """Calcule le BERTScore moyen pour un lot (batch)."""
    emb_pred = get_embeddings_batch(preds)
    emb_ref = get_embeddings_batch(refs)

    scores = []
    for i in range(len(preds)):
        p = emb_pred[i]  # (len_p, dim)
        r = emb_ref[i]   # (len_r, dim)

        # normalisation
        p = F.normalize(p, dim=1)
        r = F.normalize(r, dim=1)

        sim = torch.mm(p, r.T)

        precision = sim.max(dim=1).values.mean()
        recall = sim.max(dim=0).values.mean()

        f1 = 2 * precision * recall / (precision + recall + 1e-8)
        scores.append(f1.item())

    return sum(scores) / len(scores) if scores else 0.0

# 2. Configuration pour BLEU et ROUGE (Identique à l'équipe GPT-2)
def calculate_bleu_rouge_batch(preds, refs):
    """Calcule les scores BLEU et ROUGE moyens pour un lot."""
    bleu_scores = []
    rouge1_f1_scores = []
    rouge2_f1_scores = []
    rougel_f1_scores = []

    # Initialize RougeScorer (use_stemmer=True est important pour être aligné)
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

    # Define weights for BLEU (standard uniform weights for BLEU-4)
    bleu_weights = (0.25, 0.25, 0.25, 0.25)

    for pred, ref in zip(preds, refs):
        # Tokenize for BLEU: reference needs to be a list of token lists
        tokenized_ref = [word_tokenize(ref)]
        tokenized_pred = word_tokenize(pred)

        # Calculate BLEU score
        try:
            bleu_score = sentence_bleu(tokenized_ref, tokenized_pred, weights=bleu_weights)
            bleu_scores.append(bleu_score)
        except ZeroDivisionError:
            bleu_scores.append(0.0)

        # Calculate ROUGE scores
        scores = scorer.score(ref, pred)
        rouge1_f1_scores.append(scores['rouge1'].fmeasure)
        rouge2_f1_scores.append(scores['rouge2'].fmeasure)
        rougel_f1_scores.append(scores['rougeL'].fmeasure)

    avg_bleu = sum(bleu_scores) / len(bleu_scores) if bleu_scores else 0
    avg_rouge1_f1 = sum(rouge1_f1_scores) / len(rouge1_f1_scores) if rouge1_f1_scores else 0
    avg_rouge2_f1 = sum(rouge2_f1_scores) / len(rouge2_f1_scores) if rouge2_f1_scores else 0
    avg_rougel_f1 = sum(rougel_f1_scores) / len(rougel_f1_scores) if rougel_f1_scores else 0

    return avg_bleu, avg_rouge1_f1, avg_rouge2_f1, avg_rougel_f1

# 3. Fonction d'orchestration globale
def compute_metrics(predictions_list, references_list):
    """
    Calcule BLEU, ROUGE et BERTScore en utilisant la méthode de l'équipe GPT-2.
    """
    # Extraction de la première prédiction générée (Top-1)
    flat_preds = [preds[0] for preds in predictions_list]

    # Gestion des références multiples (on prend la première pour rester simple et aligné)
    single_refs = [refs[0] if isinstance(refs, list) else refs for refs in references_list]

    # Calcul des scores
    avg_bleu, avg_rouge1, avg_rouge2, avg_rougel = calculate_bleu_rouge_batch(flat_preds, single_refs)
    avg_bertscore = bertscore_batch(flat_preds, single_refs)

    metrics = {
        "BLEU"      : round(avg_bleu, 4),      # On garde 4 décimales comme l'équipe GPT-2
        "ROUGE-1"   : round(avg_rouge1, 4),
        "ROUGE-2"   : round(avg_rouge2, 4),
        "ROUGE-L"   : round(avg_rougel, 4),
        "BERTScore" : round(avg_bertscore, 4),
    }

    print("\nRésultats MDLM :")
    for k, v in metrics.items():
        print(f"   {k:12s} : {v:.4f}")

    return metrics

def compute_self_bleu(predictions_list):
    """
    Calcule le Self-BLEU pour mesurer la diversité des générations.
    Plus le Self-BLEU est BAS, plus les générations sont DIVERSIFIÉES.
    """
    self_bleu_scores = []
    bleu_weights = (0.25, 0.25, 0.25, 0.25)

    for preds in predictions_list:
        if len(preds) < 2:
            continue
        scores = []
        for i, hyp in enumerate(preds):
            # Les autres phrases générées servent de références
            refs = [p for j, p in enumerate(preds) if j != i]
            tokenized_refs = [word_tokenize(r) for r in refs]
            tokenized_hyp = word_tokenize(hyp)

            try:
                score = sentence_bleu(tokenized_refs, tokenized_hyp, weights=bleu_weights)
                scores.append(score)
            except ZeroDivisionError:
                scores.append(0.0)

        self_bleu_scores.append(np.mean(scores))

    mean_self_bleu = np.mean(self_bleu_scores)
    print(f"\nSelf-BLEU (diversité) : {mean_self_bleu:.4f} (↓ = plus diversifié)")
    return mean_self_bleu

Chargement des modèles pour BERTScore...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
import json
from google.colab import drive
drive.mount('/content/drive')
print("Chargement des données d'évaluation.")

# CHARGEMENT DES DONNÉES
with open("/content/drive/MyDrive/genai/eval_subset.json", "r") as f:
    eval_subset = json.load(f)
subset_to_test = eval_subset[:50]

# GÉNÉRATION
print(f"Génération MDLM en cours pour {len(subset_to_test)} exemples.")
predictions = generate_descriptions_mdlm(
    subset_to_test,
    n_samples=3,        # 3 descriptions pour pouvoir calculer le Self-BLEU
    max_new_tokens=128,
    steps=32,
    block_size=64,
    temperature=0.5     # Le sample 0 sera à 0.0, les samples 1 et 2 seront à 0.5
)

# PRÉPARATION DES RÉFÉRENCES
references = [[ex["reference"]] for ex in subset_to_test]

# CALCUL DES MÉTRIQUES
print("\nCalcul des métriques.")
metrics = compute_metrics(predictions, references)
self_bleu = compute_self_bleu(predictions)

# SAUVEGARDE DES RÉSULTATS
results = {
    "predictions": predictions,
    "metrics": metrics,
    "self_bleu": self_bleu
}

# On sauvegarde sur le Drive
output_path = "/content/drive/MyDrive/genai/mdlm_results.json"
with open(output_path, "w") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"\nRésultats sauvegardés dans : {output_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Chargement des données d'évaluation.
Génération MDLM en cours pour 50 exemples.
